# Analyse test SCIO data

## Rationale

To calculate the absorbance spectrum, use the following formula:

$A = -log10((S - D - G) / (SW - SWD - SWG))$

Where:

- A is the absorbance spectrum
- S is the sample spectrum
- D is the sample dark spectrum
- G is the sample gradient spectrum
- SW is the sample white spectrum
- SWD is the sample white dark spectrum
- SWG is the sample white gradient spectrum

The subtractions and divisions above are part of a common data preprocessing step in spectrophotometry called baseline correction. The idea is to remove any contributions to the measured signal that are not due to the sample itself, but rather due to the instrument or the surrounding environment. This helps to enhance the accuracy and reliability of the measurements, particularly in the presence of noise or other sources of variability.

The specific steps involved in baseline correction can vary depending on the particular instrument and experimental setup, but some common approaches include:

- Subtracting a background or reference spectrum, such as a dark spectrum or a blank sample, to remove any contributions from the instrument or the surrounding environment.
- Dividing the sample spectrum by a reference spectrum, such as a white or standard spectrum, to correct for any variations in the intensity or wavelength response of the instrument.
- Applying mathematical transformations, such as smoothing or differentiation, to remove any high-frequency noise or artifacts in the data.

In the specific example given above, the baseline correction involves subtracting the dark and white spectra to remove any contributions from the instrument or the environment, and then dividing the sample and white spectra to correct for any variations in the instrument response. The resulting absorbance spectrum should represent the contribution of the sample alone, without any interference from other sources.

The gradient spectrum is a measurement of how the intensity of the light changes over the wavelength range. It is typically measured by taking a set of measurements with a varying concentration of a sample, which allows one to calculate the slope of the absorbance vs. concentration curve at each wavelength. This is useful for determining the sensitivity of the spectrometer at different wavelengths, as well as for correcting for changes in the intensity of the light source over time. The gradient spectrum can be subtracted from the sample spectrum to correct for any changes in the intensity of the light source during the measurement.

In [ ]:
import numpy as np

def calculate_absorbance_spectrum(sample, sample_dark, sample_gradient, sample_white, sample_white_dark, sample_white_gradient):
    # Subtract dark and gradient from sample
    S = np.subtract(sample, sample_dark)
    S = np.subtract(S, sample_gradient)

    # Subtract dark and gradient from sample white
    SW = np.subtract(sample_white, sample_white_dark)
    SW = np.subtract(SW, sample_white_gradient)

    # Calculate absorbance spectrum
    A = -np.log10(np.divide(S, SW))

    return A

# This function first subtracts the dark and gradient spectra from the sample spectrum and the sample white spectrum.
# It then calculates the absorbance spectrum using the formula above and returns it as output. Note that this
# implementation assumes that the input spectra are numpy

## Data extraction from raw

- I have 331 values, and normally an int is 4 bytes, so should be 1324 bytes.
- The sample and sample_dark contain 1800 bytes, the sample_gradient contains 1656 bytes
- Gradient:
  - If there is some sort of padding byte, then we get $331*4 + 331*1 = 1655$ bytes, containing 1 length byte (the first one)
  - Alternatively, the first or last 331 bytes could be a different message of something
- Main data:
  - There are 4 empty padding bytes. Then we have 1796 bytes, which is not dividable by 16, i.e. could not be encrypted data. Unless the next few bytes are encrypted too?

In [ ]:
print((1800-4) - 331*5)

# Assuming encryption, both types have the first 8 bytes superfluous
print(112*16)
print(103*16)

In [ ]:
import json

# Could the device ID be the key?
# Load JSON data from file
with open('test_data.json') as f:
    data = json.load(f)
#print(data)
print(data['device_id'])
key = bytes(data['device_id'], 'latin-1')
print(key)

byte_key = bytes.fromhex(data['device_id'])
print(len(byte_key), 'bytes key length, not enough as a key')

In [ ]:
import base64
from Crypto.Cipher import AES

# Get the base64 string from the 'sample' dataset
b64_string = data['sample']

# Decode the base64 string into bytes
bytes_data = base64.b64decode(b64_string)

key = bytes.fromhex('00004c8e3238c8e18032ab45611198f1')
# Create an AES cipher object with CTR mode
cipher = AES.new(key, AES.MODE_CTR)

# Decrypt the data
decrypted_data = cipher.decrypt(bytes_data[8:])

# Print the result
#print(decrypted_data)
print(len(decrypted_data))
print(decrypted_data.hex())

import struct
# assume your bytes are in a variable called `data`
start = 0  # skip the first 1 byte
num_values = 331
value_size = 4  # each value is 4 bytes
separator_size = 1  # there is 1 byte separating each value
fmt = f">{separator_size}x{num_values}I"  # use 'x' to skip the separator byte

fmt=f'<LLLL'

int_data = list(struct.unpack_from(fmt, bytes_data, start))
#int_data = [x * 10**(-9) for x in int_data]

# Print the first 10 integers
print(int_data)

In [ ]:
import json
import base64

with open('test_result.json') as f:
    res = json.load(f)
print(res['spectrum'][:10])

# Load JSON data from file
with open('test_data.json') as f:
    data = json.load(f)
    
print(data['device_id'])

# Get the base64 string from the 'sample' dataset
b64_string = data['sample']

# Decode the base64 string into bytes
bytes_data = base64.b64decode(b64_string)

print(len(bytes_data))
#print(bytes_data)


import struct
# assume your bytes are in a variable called `data`
start = 0  # skip the first 1 byte
num_values = 331
value_size = 4  # each value is 4 bytes
separator_size = 1  # there is 1 byte separating each value
fmt = f">{separator_size}x{num_values}I"  # use 'x' to skip the separator byte

fmt = f">IIIBIBIBIBIBI"

int_data = list(struct.unpack_from(fmt, bytes_data, start))
print(len(int_data))
#int_data = [x * 10**(-9) for x in int_data]

# Print the first 10 integers
print(int_data)

In [ ]:
from Crypto.Cipher import AES
import base64

# Load JSON data from file
with open('test_data.json') as f:
    encrypted_data = json.load(f)

# The decryption key
key = "8032AB45611198F1".encode('latin-1')

# Create an AES cipher object with the given key and ECB mode
cipher = AES.new(key, AES.MODE_ECB)

# Decode the base64 string to bytes
encrypted_bytes = base64.b64decode(encrypted_data['sample'])

# Decrypt the data
decrypted_bytes = cipher.decrypt(encrypted_bytes[4:]) # Slice to remove padding

# Convert the bytes to a list of integers
int_list = [int.from_bytes(decrypted_bytes[i:i+4], 'big', signed=False) for i in range(0, len(decrypted_bytes), 4)]

In [ ]:
# Load JSON data from file
with open('test_data.json') as f:
    data = json.load(f)

# Get the base64 string from the 'sample' dataset
b64_string = data['sample_white_gradient']

# Decode the base64 string into bytes
bytes_data = base64.b64decode(b64_string)

print(len(bytes_data))

import struct
# assume your bytes are in a variable called `data`
start = 6  # skip the first 6 bytes
num_values = 330
value_size = 4  # each value is 4 bytes
separator_size = 1  # there is 1 byte separating each value
fmt = f">{num_values}I{separator_size}x"  # use 'x' to skip the separator byte

int_data_w = struct.unpack_from(fmt, bytes_data, start)

int_data_w = [x * 10**(-10) for x in int_data_w]

# Print the first 10 integers
print(int_data_w[:10])

In [ ]:
331*5

In [ ]:
result = [int_data_w[i] - int_data[i] for i in range(len(int_data))]

print(result)

In [ ]:
import numpy as np

# Read raw data from file or device
raw_data = [0, 0, 0, 1, 2, 3, 4, 5, 4, 3, 2, 1, 0, 0, 0]

# Define wavelength range
wavelengths = np.linspace(740, 1070, 331)
print(wavelengths)

# Subtract dark spectrum (optional)
dark_spectrum = [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]
processed_data = [raw - dark for raw, dark in zip(raw_data, dark_spectrum)]

# Normalize to white spectrum (optional)
white_spectrum = [0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 8, 8, 8, 8]
processed_data = [raw / white for raw, white in zip(processed_data, white_spectrum)]

# Convert to absorbance
reference_spectrum = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1]
absorbance_spectrum = [-np.log10(raw / reference) for raw, reference in zip(processed_data, reference_spectrum)]


In [ ]:
103*16